In [1]:
import pandas as pd

# Detailed Listing Explore(Detailed/listing.csv)

In [2]:
detailedListing_df = pd.read_csv("Detailed/listings.csv")
detailedReview_df = pd.read_csv("Detailed/reviews.csv")
detailedCalander_df = pd.read_csv("Detailed/calendar.csv")
neighbourhood_df = pd.read_csv("neighbourhoods.csv")

In [3]:
detailedListing_df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,10803,https://www.airbnb.com/rooms/10803,20260616211523,2026-06-17,city scrape,"Room in Deco Apartment, Brunswick East",A large air conditioned room with firm queen s...,NaN,https://a0.muscache.com/pictures/e5f30dd1-ac57...,38901,...,4.73,4.70,4.66,NaN,NaN,1,0,1,0,1.31
1,12936,https://www.airbnb.com/rooms/12936,20260616211523,2026-06-28,previous scrape,St Kilda 1BR+BEACHSIDE+BALCONY+WIFI+AC,RIGHT IN THE HEART OF ST KILDA! It doesn't get...,NaN,https://a0.muscache.com/pictures/59701/2e8cdaf...,50121,...,4.83,4.78,4.66,NaN,NaN,10,10,0,0,0.22
2,41836,https://www.airbnb.com/rooms/41836,20260616211523,2026-06-28,previous scrape,CLOSE TO CITY & MELBOURNE AIRPORT,Easy to travel from and to the Airport; quiet ...,NaN,https://a0.muscache.com/pictures/569696dd-1ad0...,182833,...,4.83,4.39,4.69,NaN,NaN,2,0,2,0,0.83
3,43429,https://www.airbnb.com/rooms/43429,20260616211523,2026-06-17,city scrape,Tranquil Javanese Studio and Pond!,"No service/Cleaning Fees, EV Charger, Study th...",NaN,https://a0.muscache.com/pictures/airflow/Hosti...,189684,...,4.94,4.79,4.86,NaN,NaN,2,2,0,0,1.48
4,44699,https://www.airbnb.com/rooms/44699,20260616211523,2026-06-17,city scrape,"15 yearsHosting (4.8), 8 CITY TRAMS, GymPoolTe...",Unwavering service — just ask and we do our be...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,189245,...,4.97,4.84,4.71,NaN,NaN,1,0,1,0,0.35


In [4]:
x = detailedListing_df['price'].replace("$", "")

In [5]:
x

0         $60.74
1            NaN
2            NaN
3        $165.00
4        $215.00
          ...   
25723    $100.00
25724    $197.50
25725    $364.40
25726        NaN
25727    $127.48
Name: price, Length: 25728, dtype: str

In [6]:
detailedListing_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25728 entries, 0 to 25727
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            25728 non-null  int64  
 1   listing_url                                   25728 non-null  str    
 2   scrape_id                                     25728 non-null  int64  
 3   last_scraped                                  25728 non-null  str    
 4   source                                        25728 non-null  str    
 5   name                                          25728 non-null  str    
 6   description                                   25276 non-null  str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   25728 non-null  str    
 9   host_id                                       25728 non-null  int64  
 1

In [7]:
detailedListing_schema_df = detailedListing_df.dtypes.reset_index()

detailedListing_schema_df.columns = ["Column Name", "Data Type"]

In [8]:
detailedListing_schema_df.head()

,Column Name,Data Type
0,id,int64
1,listing_url,str
2,scrape_id,int64
3,last_scraped,str
4,source,str


In [9]:
def extract(df) -> pd.DataFrame:

    Detailed_Data = []
    
    for col in df.columns:
        dtype = df[col].dtype
    
        sample_value = df[col].dropna().unique()[:1].tolist()
        is_valid = pd.api.types.is_numeric_dtype(dtype) or pd.api.types.is_datetime64_any_dtype(dtype)
    
        Detailed_Data.append({
            'Column Name': col,
            'Data Type': dtype,
            'Min Value': df[col].min() if is_valid else 'N/A',
            'Max Value': df[col].max() if is_valid else 'N/A',
            'Sample Values': sample_value
        })
    
    return pd.DataFrame(Detailed_Data)

In [10]:
detailedListing_subset_df = extract(detailedListing_df)
detailedReview_subset_df = extract(detailedReview_df)
detailedCalander_subset_df = extract(detailedCalander_df)
#neighbourhood_df = extract(neighbourhood_df)

In [11]:
with pd.ExcelWriter("airbnb_melbourne_schema_documentation.xlsx", engine = "openpyxl") as writer:
    detailedListing_subset_df.to_excel(writer, sheet_name = "Listing Schema", index = False)
    detailedReview_subset_df.to_excel(writer, sheet_name = "Review Schema", index = False)
    detailedCalander_subset_df.to_excel(writer, sheet_name = "Calander Schema", index = False)

## Finding the Relationships & PK and Fks

In [12]:
dataframes = {
    'listing': detailedListing_df,
    'review': detailedReview_df,
    'calander': detailedCalander_df,
}


In [13]:
import pandas as pd

def profile_pk_fk(dfs: dict[str, pd.DataFrame]):
    summary = {}
    
    # 1. Identify Primary Key Candidates
    for name, df in dfs.items():
        total_rows = len(df)
        pks = [col for col in df.columns if df[col].nunique() == total_rows and total_rows > 0]
        summary[name] = {"pk_candidates": pks, "fk_candidates": []}
    
    # 2. Identify Foreign Key Candidates
    for table_a, df_a in dfs.items():
        for table_b, df_b in dfs.items():
            if table_a == table_b:
                continue
                
            # Check if columns in Table B match a confirmed PK candidate in Table A
            for pk in summary[table_a]["pk_candidates"]:
                for col_b in df_b.columns:
                    # Match by name convention and data subset
                    if pk in col_b or col_b in pk:
                        set_a = set(df_a[pk].dropna())
                        set_b = set(df_b[col_b].dropna())
                        
                        # If B is a subset of A, it's a Foreign Key
                        if set_b.issubset(set_a) and len(set_b) > 0:
                            summary[table_b]["fk_candidates"].append({
                                "column": col_b,
                                "references_table": table_a,
                                "references_column": pk
                            })
                            
    return summary

In [14]:
profile_pk_fk(dataframes)

{'listing': {'pk_candidates': ['id', 'listing_url'], 'fk_candidates': []},
 'review': {'pk_candidates': ['id'],
  'fk_candidates': [{'column': 'listing_id',
    'references_table': 'listing',
    'references_column': 'id'}]},
 'calander': {'pk_candidates': [],
  'fk_candidates': [{'column': 'listing_id',
    'references_table': 'listing',
    'references_column': 'id'}]}}

In [15]:
detailedCalander_df.head()

,listing_id,date,available,minimum_nights,maximum_nights
0,10803,2026-06-17,f,12,28
1,10803,2026-06-18,f,12,28
2,10803,2026-06-19,t,12,28
3,10803,2026-06-20,t,12,28
4,10803,2026-06-21,t,12,28


In [16]:
detailedCalander_df['listing_id'].is_unique

False

In [17]:
detailedCalander_df['date'].is_unique

False

In [18]:
composite_cols = ['listing_id', 'date']
is_composite_unique = not detailedCalander_df.duplicated(subset=composite_cols).any()

In [19]:
is_composite_unique

True

In [20]:
detailedListing_df.dtypes.value_counts()

float64    38
str        28
int64      24
Name: count, dtype: int64

# Pipeline cleaned dataset examin

In [21]:
mallorca_raw = pd.read_csv("/home/naviya-c/Desktop/dataProject/stage_3/data/01_raw/mallorca/2026-06-23/listing.csv")

In [22]:
mallorca_cleaned = pd.read_csv("/home/naviya-c/Desktop/dataProject/stage_3/data/02_clean/mallorca/2026-06-23/listing_clean.csv")

In [23]:
mallorca_cleaned.head(4)

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,1594759156338136230,https://www.airbnb.com/rooms/1594759156338136230,20260623040707,2026-06-23,city scrape,S'Hortet Gran,Arrive and feel at home.,0.0,https://a0.muscache.com/pictures/hosting/Hosti...,141515273,...,0.0,0.00,0.00,Mallorca - Regional registration number<br />E...,0.0,48,48,0,0,0.00
1,1273823517224415510,https://www.airbnb.com/rooms/1273823517224415510,20260623040707,2026-06-23,city scrape,Son Pere Genet,ETV 411 Recharge your batteries in this wonder...,0.0,https://a0.muscache.com/pictures/miso/Hosting-...,651077862,...,5.0,4.86,4.86,Spain - National registration number<br />ESFC...,0.0,1,1,0,0,0.47
2,1151380048474115597,https://www.airbnb.com/rooms/1151380048474115597,20260623040707,2026-06-23,city scrape,Ca Na Reieta by Vintage Travel,Ground Floor: Open plan living/dining room (TV...,0.0,https://a0.muscache.com/pictures/prohost-api/H...,565714005,...,0.0,0.00,0.00,Mallorca - Regional registration number<br />E...,0.0,40,40,0,0,0.00
3,1430444173039950930,https://www.airbnb.com/rooms/1430444173039950930,20260623040707,2026-06-23,city scrape,Villa Sofía,Vacation villa for 8 people with fenced pool a...,0.0,https://a0.muscache.com/pictures/miso/Hosting-...,29095214,...,0.0,0.00,0.00,Spain - National registration number<br />ESFC...,0.0,24,24,0,0,0.00


In [24]:
mallorca_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 14817 entries, 0 to 14816
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            14817 non-null  int64  
 1   listing_url                                   14817 non-null  str    
 2   scrape_id                                     14817 non-null  int64  
 3   last_scraped                                  14817 non-null  str    
 4   source                                        14817 non-null  str    
 5   name                                          14817 non-null  str    
 6   description                                   14529 non-null  str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   14817 non-null  str    
 9   host_id                                       14817 non-null  int64  
 1

In [25]:
mallorca_cleaned['price'].head()

0      0.00
1    319.60
2    201.43
3    330.71
4    554.50
Name: price, dtype: float64

In [26]:
mallorca_raw['price'].head()

0        NaN
1    $319.60
2    $201.43
3    $330.71
4    $554.50
Name: price, dtype: str

In [71]:
mallorca_raw['last_scraped'].head()

0    2026-06-23
1    2026-06-23
2    2026-06-23
3    2026-06-23
4    2026-06-23
Name: last_scraped, dtype: str

In [57]:
mallorca_enrich = pd.read_csv("/home/naviya-c/Desktop/dataProject/stage_3/data/03_enriched/mallorca/2026-06-23/listing_master.csv")

In [58]:
mallorca_enrich_col = mallorca_enrich.columns.tolist()

In [59]:
len(mallorca_enrich_col)

100

In [60]:
mallorca_raw_col = mallorca_raw.columns.tolist()

In [61]:
len(mallorca_raw_col)

90

In [62]:
add_columns = list(set(mallorca_enrich_col) - set(mallorca_raw_col))

In [63]:
add_columns

['price_per_bedroom',
 'total_days',
 'host_tenure_years',
 'neighbourhood_listing_count',
 'estimated_revenue',
 'neighbourhood_average_rating',
 'neighbourhood_median_price',
 'occupied_days',
 'review_frequency',
 'occupancy_rate']

In [37]:
mallorca_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 14817 entries, 0 to 14816
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            14817 non-null  int64  
 1   listing_url                                   14817 non-null  str    
 2   scrape_id                                     14817 non-null  int64  
 3   last_scraped                                  14817 non-null  str    
 4   source                                        14817 non-null  str    
 5   name                                          14817 non-null  str    
 6   description                                   14529 non-null  str    
 7   neighborhood_overview                         14817 non-null  float64
 8   picture_url                                   14817 non-null  str    
 9   host_id                                       14817 non-null  int64  
 1

In [69]:
mallorca_enrich['host_since']

0        1970-01-01
1        1970-01-01
2        1970-01-01
3        1970-01-01
4        1970-01-01
            ...    
14812    1970-01-01
14813    1970-01-01
14814    1970-01-01
14815    1970-01-01
14816    1970-01-01
Name: host_since, Length: 14817, dtype: str